# Probe-protocol queue, two cards: fp64-pairlab

10 runs (compile_pairlat_f32, compile_pairlat_fp64, pairlab_f32, pairlab_fp64) on the probe protocol (K=32, 400k training jets, 20 epochs),
the 2026-10-05 control's command plus each run's flags, drained by two T4s
as a queue. T4 with torch 2.11.0+cu128 pinned and asserted. Kernel budget 2.6 h.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
TAG, JOBS, BUDGET = 'fp64-pairlab', [['pairlab_fp64', ['--pair-lab', '--pair-lab-fp64'], 0], ['pairlab_f32', ['--pair-lab'], 0], ['pairlab_fp64', ['--pair-lab', '--pair-lab-fp64'], 1], ['pairlab_f32', ['--pair-lab'], 1], ['pairlab_fp64', ['--pair-lab', '--pair-lab-fp64'], 2], ['pairlab_f32', ['--pair-lab'], 2], ['compile_pairlat_fp64', ['--pair-lab', '--pair-lab-fp64', '--pair-latent', '8', '--epochs', '1', '--max-train-samples', '20000'], 0], ['compile_pairlat_f32', ['--pair-lab', '--pair-latent', '8', '--epochs', '1', '--max-train-samples', '20000'], 0], ['compile_pairlat_fp64', ['--pair-lab', '--pair-lab-fp64', '--pair-latent', '8', '--epochs', '1', '--max-train-samples', '20000'], 1], ['compile_pairlat_f32', ['--pair-lab', '--pair-latent', '8', '--epochs', '1', '--max-train-samples', '20000'], 1]], 2.6 * 3600
OUT = "/kaggle/working/results_probe_queue"
CKPT = "/kaggle/working/checkpoints"
LOGS = "/kaggle/working/logs/" + TAG
RUN_TIME = 0.55 * 3600       # one probe run end to end, two cards busy (0.49-0.60 h for pair_lab)
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--max-train-samples', '400000', '--epochs', '20', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '32', '--eval-chunk-size', '2048', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
for d in (OUT, LOGS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
pathlib.Path(OUT, "bundle_sha256.txt").write_text('cae2e74a180e' + chr(10))
print("data:", DATA, "| bundle", 'cae2e74a180e', "|", len(JOBS), "jobs")

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# Smoke on both cards at once with the first job's flags: one epoch on 20k
# jets per card. Each card must write a result to be given jobs.
import glob
smoke = []
for i in range(NGPU):
    d = "/kaggle/working/smoke/%s/card%d" % (TAG, i)
    smoke.append((d, launch(["benchmarks.run_top_tagging"] + BASE + JOBS[0][1] +
                            ["--epochs", "1", "--max-train-samples", "20000",
                             "--seed", "0", "--results-dir", d],
                            i, LOGS + "/smoke_card%d.log" % i)))
codes = [p.wait() for _, p in smoke]
CARDS = [i for i, ((d, _), rc) in enumerate(zip(smoke, codes)) if glob.glob(d + "/*.json")]
for i, rc in enumerate(codes):
    print("card %d: exit %d, result %s" % (i, rc, i in CARDS))
    if i not in CARDS:
        print(tail(LOGS + "/smoke_card%d.log" % i))
assert CARDS, "the smoke run failed on every card"
print("cards in use:", CARDS)

In [ ]:
# The queue: each free card takes the next job; no job starts once less than
# one run's time is left of the budget.
import glob, time
todo = list(JOBS)
running = {}                     # card -> (arm, seed, process)
BAD, SKIPPED = [], []
while todo or running:
    for c in CARDS:
        if c in running:
            arm, s, p = running[c]
            if p.poll() is None and time.time() - T0 > BUDGET + 0.1 * 3600:
                p.terminate(); p.wait()          # hung past the budget: its checkpoint is kept
                print("card %d: %s seed %d terminated past the budget" % (c, arm, s), flush=True)
            if p.poll() is None:
                continue
            ok = p.returncode == 0 and glob.glob(OUT + "/%s/seed%d/*.json" % (arm, s))
            print("%7.2f h  card %d: %s seed %d %s" % ((time.time() - T0) / 3600, c, arm, s,
                                                       "done" if ok else "FAILED, exit %d" % p.returncode), flush=True)
            if not ok:
                BAD.append((arm, s)); print(tail(LOGS + "/%s_seed%d.log" % (arm, s), 1500))
            del running[c]
        if todo and c not in running:
            if time.time() - T0 + RUN_TIME > BUDGET:
                SKIPPED += [(a, s) for a, _, s in todo]; todo = []
                print("budget: the remaining jobs are skipped:", SKIPPED); continue
            arm, flags, s = todo.pop(0)
            cap = int(min(RUN_TIME * 1.5, BUDGET - (time.time() - T0) - 600))
            running[c] = (arm, s, launch(["benchmarks.run_top_tagging"] + BASE + flags +
                                         ["--seed", str(s), "--results-dir", OUT + "/%s/seed%d" % (arm, s),
                                          "--ckpt-dir", CKPT + "/%s_seed%d" % (arm, s), "--resume",
                                          "--max-seconds", str(cap)],
                                         c, LOGS + "/%s_seed%d.log" % (arm, s)))
            print("%7.2f h  card %d: start %s seed %d" % ((time.time() - T0) / 3600, c, arm, s), flush=True)
    time.sleep(20)
print("failed:", BAD or "none", "| skipped:", SKIPPED or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# The float64 pair features against float32, both from this kernel, three
# seeds each, read by the rule fixed before the push: d = mean(fp64) -
# mean(f32); "pays" if d >= +0.00015 and Welch p < 0.05, "hurts" if
# d <= -0.00015 and p < 0.05, otherwise no resolved difference. Beside it, the
# earlier float32 --pair-lab probe runs pooled with this kernel's, the probe
# controls, and the compile check of every run.
import json, glob, statistics as st, math
from scipy import stats
REFS, RULE = {'pairlab': [0.9814375514952488, 0.9816360003720188, 0.9817201289163335, 0.9812255429282452], 'control': [0.9811491732570162, 0.9811092523841787, 0.981106801276, 0.9809738936815733, 0.9810829106236938, 0.9810854817813994, 0.9809025757586377]}, (0.00015, 0.05)
def load(arm, epochs=20):
    rs = [json.load(open(f)) for f in sorted(glob.glob(OUT + "/%s/seed*/*.json" % arm))]
    for r in rs:
        t = r["test_metrics"]; env = r["env"]; cc = r.get("compile_check") or {}
        assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
        print("%-21s seed %d: AUC %.5f  rej@0.3 %.1f  %d epochs  %.2f h  compile %s d_logit %.1e  kwargs %s"
              % (arm, r["seed"], t["test_auc"], t["bg_rej_30"], r["epochs_run"], r["walltime_sec"] / 3600,
                 cc.get("used"), cc.get("d_logit", float("nan")), r["so3c_kwargs"]))
    return {r["seed"]: r["test_metrics"]["test_auc"] for r in rs if r["epochs_run"] == epochs}
def welch(a, b):
    va, vb = st.variance(a) / len(a), st.variance(b) / len(b)
    t = (st.mean(a) - st.mean(b)) / math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    return st.mean(a) - st.mean(b), t, df, 2 * stats.t.sf(abs(t), df)
fp, f32 = load("pairlab_fp64"), load("pairlab_f32")
for arm in ("compile_pairlat_fp64", "compile_pairlat_f32"):
    load(arm, epochs=1)
a, b = list(fp.values()), list(f32.values())
if len(a) >= 2 and len(b) >= 2:
    d, t, df, p = welch(a, b)
    if d >= RULE[0] and p < RULE[1]:
        verdict = "PAYS: carry --pair-lab-fp64 into later runs, with same-stack references"
    elif d <= -RULE[0] and p < RULE[1]:
        verdict = "HURTS: keep float32"
    else:
        verdict = "no resolved difference"
    print("fp64 %d runs %.5f +- %.5f | f32 here %d runs %.5f +- %.5f | d %+.5f (Welch t %.2f, df %.1f, p %.3f) -> %s"
          % (len(a), st.mean(a), st.stdev(a), len(b), st.mean(b), st.stdev(b), d, t, df, p, verdict))
    same = sorted(set(fp) & set(f32))
    print("same-seed differences (both runs started together):", " ".join("s%d %+.5f" % (s, fp[s] - f32[s]) for s in same))
    pooled = REFS["pairlab"] + b
    d2, t2, df2, p2 = welch(a, pooled)
    print("against all %d float32 --pair-lab probe runs (%.5f +- %.5f): %+.5f (t %.2f, df %.1f, p %.3f)"
          % (len(pooled), st.mean(pooled), st.stdev(pooled), d2, t2, df2, p2))
    for name, rows in (("f32 here", b), ("fp64", a)):
        d3, t3, df3, p3 = welch(rows, REFS["control"])
        print("%-8s against the %d probe controls (%.5f): %+.5f (t %.1f, p %.3f)"
              % (name, len(REFS["control"]), st.mean(REFS["control"]), d3, t3, p3))
    print("float32 --pair-lab here against the earlier float32 runs: %+.5f (kernel offset)"
          % (st.mean(b) - st.mean(REFS["pairlab"])))
else:
    print("fewer than two complete runs per arm: no reading")
assert not BAD, "jobs failed: %s" % BAD